In [1]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import math
import os
from sklearn.linear_model import SGDRegressor, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.metrics import mean_squared_error, accuracy_score
import warnings
warnings.simplefilter('ignore')

In [2]:
class BatchGDRegression:
    def __init__(self):
        self.intercept_ = 0.0
        self.coef_ = []

    def fit(self, x, y, learningRate=0.01, noEpochs=1000):
        self.coef_ = [0.0 for _ in range(len(x[0]))]
        self.intercept_ = 0.0
        n_samples = len(x)

        for epoch in range(noEpochs):
            gradients_w = [0.0 for _ in range(len(x[0]))]
            gradient_b = 0.0

            # Calcularea erorilor si a gradientilor pe tot setul de date
            for i in range(n_samples):
                ycomputed = self.eval(x[i])
                crtError = ycomputed - y[i]

                for j in range(len(x[0])):
                    gradients_w[j] += crtError * x[i][j]
                gradient_b += crtError

            # Actualizarea coeficientilor si a interceptului folosind media gradientilor
            for j in range(len(x[0])):
                self.coef_[j] -= learningRate * gradients_w[j] / n_samples
            self.intercept_ -= learningRate * gradient_b / n_samples

    def eval(self, xi):
        yi = self.intercept_
        for j in range(len(xi)):
            yi += self.coef_[j] * xi[j]
        return yi

    def predict(self, x):
        return [self.eval(xi) for xi in x]

In [4]:
inputs_gdp = []
inputs_freedom = []
outputs_happiness = []

with open('C:/Laborator AI/Laborator 6/world-happiness-report-2017.csv', 'r') as file:
    csv_reader = csv.reader(file)
    header = next(csv_reader)

    for row in csv_reader:
        outputs_happiness.append(float(row[2])) # Happiness.Score
        inputs_gdp.append(float(row[5]))        # Economy..GDP.per.Capita.
        inputs_freedom.append(float(row[8]))    # Freedom

inputs_gdp = np.array(inputs_gdp)
inputs_freedom = np.array(inputs_freedom)
outputs_happiness = np.array(outputs_happiness)

X_1_feature = [[val] for val in inputs_gdp]
X_2_features = [[inputs_gdp[i], inputs_freedom[i]] for i in range(len(inputs_gdp))]

# Scalare date
scaler1 = StandardScaler()
X_1_scaled = scaler1.fit_transform(X_1_feature)
scaler2 = StandardScaler()
X_2_scaled = scaler2.fit_transform(X_2_features)

print("--- Regresie doar cu GDP ---")
# Tool
reg_tool_1 = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.01)
reg_tool_1.fit(X_1_scaled, outputs_happiness)
print("MSE Tool (1 feature):", mean_squared_error(outputs_happiness, reg_tool_1.predict(X_1_scaled)))

# Cod propriu
reg_custom_1 = BatchGDRegression()
reg_custom_1.fit(X_1_scaled, outputs_happiness, learningRate=0.1, noEpochs=1000)
print("MSE Custom Batch GD (1 feature):", mean_squared_error(outputs_happiness, reg_custom_1.predict(X_1_scaled)))

print("\n--- Regresie cu GDP si Freedom ---")
# Tool
reg_tool_2 = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.01)
reg_tool_2.fit(X_2_scaled, outputs_happiness)
print("MSE Tool (2 features):", mean_squared_error(outputs_happiness, reg_tool_2.predict(X_2_scaled)))

# Cod propriu
reg_custom_2 = BatchGDRegression()
reg_custom_2.fit(X_2_scaled, outputs_happiness, learningRate=0.1, noEpochs=1000)
print("MSE Custom Batch GD (2 features):", mean_squared_error(outputs_happiness, reg_custom_2.predict(X_2_scaled)))

--- Regresie doar cu GDP ---
MSE Tool (1 feature): 0.43248410922397096
MSE Custom Batch GD (1 feature): 0.4321505672954764

--- Regresie cu GDP si Freedom ---
MSE Tool (2 features): 0.3258646484470811
MSE Custom Batch GD (2 features): 0.3250706060454289


In [5]:
def sigmoid(x):
    # Overflow
    if x < -709: return 0.0
    if x > 709: return 1.0
    return 1 / (1 + math.exp(-x))

class CustomLogisticRegression:
    def __init__(self):
        self.intercept_ = 0.0
        self.coef_ = []

    def fit(self, x, y, learningRate=0.01, noEpochs=1000):
        self.coef_ = [0.0 for _ in range(len(x[0]))]
        self.intercept_ = 0.0
        n_samples = len(x)

        for epoch in range(noEpochs):
            gradients_w = [0.0 for _ in range(len(x[0]))]
            gradient_b = 0.0

            for i in range(n_samples):
                ycomputed = sigmoid(self.eval(x[i]))
                crtError = ycomputed - y[i]

                for j in range(len(x[0])):
                    gradients_w[j] += crtError * x[i][j]
                gradient_b += crtError

            for j in range(len(x[0])):
                self.coef_[j] -= learningRate * gradients_w[j] / n_samples
            self.intercept_ -= learningRate * gradient_b / n_samples

    def eval(self, xi):
        yi = self.intercept_
        for j in range(len(xi)):
            yi += self.coef_[j] * xi[j]
        return yi

    def predict(self, x, threshold=0.5):
        predictions = []
        for xi in x:
            prob = sigmoid(self.eval(xi))
            predictions.append(1 if prob >= threshold else 0)
        return predictions

In [6]:
print("--- Clasificarea leziunilor (Benign / Malign) ---")
X_cancer = []
y_cancer = []

with open('C:/Laborator AI/Laborator 6/wdbc.data', 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        if not row: continue
        y_cancer.append(0 if row[1] == 'M' else 1)
        X_cancer.append([float(row[2]), float(row[3])])

X_cancer = np.array(X_cancer)
y_cancer = np.array(y_cancer)

scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

lesion = scaler_cancer.transform([[18.0, 10.0]])

# Tool
clf_tool = LogisticRegression()
clf_tool.fit(X_cancer_scaled, y_cancer)
pred_tool = clf_tool.predict(lesion)[0]
target_name = 'malignant' if pred_tool == 0 else 'benign'
print(f"Tool predictie leziune: {target_name}")

# Cod Propriu
clf_custom = CustomLogisticRegression()
clf_custom.fit(X_cancer_scaled.tolist(), y_cancer.tolist(), learningRate=0.1, noEpochs=1000)
pred_custom = clf_custom.predict(lesion.tolist())[0]
target_name_custom = 'malignant' if pred_tool == 0 else 'benign'
print(f"Custom predictie leziune: {target_name_custom}")

--- Clasificarea leziunilor (Benign / Malign) ---
Tool predictie leziune: malignant
Custom predictie leziune: malignant


In [7]:
class CustomMultiClassLogisticRegression:
    def __init__(self):
        self.models = []

    def fit(self, X, y, learningRate=0.01, noEpochs=1000):
        self.models = []
        classes = list(set(y))
        for cls in classes:
            # Transformam problema intr-una binara: clasa curenta vs restul
            binary_y = [1 if val == cls else 0 for val in y]
            model = CustomLogisticRegression()
            model.fit(X, binary_y, learningRate, noEpochs)
            self.models.append((cls, model))

    def predict(self, X):
        predictions = []
        for xi in X:
            probs = []
            for cls, model in self.models:
                probs.append((cls, sigmoid(model.eval(xi))))
            # Alegem clasa cu probabilitatea maxima
            best_cls = max(probs, key=lambda item: item[1])[0]
            predictions.append(best_cls)
        return predictions

In [8]:
print("\n--- Clasificarea speciilor de Iris ---")
X_iris = []
y_iris = []
class_mapping = {"Iris-setosa": 0, "Iris-versicolor": 1, "Iris-virginica": 2}
target_names = ["setosa", "versicolor", "virginica"]

with open('C:/Laborator AI/Laborator 6/iris.data', 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        if not row or len(row) < 5: continue
        X_iris.append([float(x) for x in row[:4]])
        y_iris.append(class_mapping[row[4].strip()])

X_iris = np.array(X_iris)
y_iris = np.array(y_iris)

scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

flower = scaler_iris.transform([[5.35, 3.85, 1.25, 0.4]])

# Tool
clf_iris_tool = LogisticRegression()
clf_iris_tool.fit(X_iris_scaled, y_iris)
pred_iris_tool = clf_iris_tool.predict(flower)[0]
print(f"Tool predictie specie Iris: {target_names[pred_iris_tool]}")

# Cod Propriu
clf_iris_custom = CustomMultiClassLogisticRegression()
clf_iris_custom.fit(X_iris_scaled.tolist(), y_iris.tolist(), learningRate=0.1, noEpochs=1000)
pred_iris_custom = clf_iris_custom.predict(flower.tolist())[0]
print(f"Custom predictie specie Iris: {target_names[pred_iris_custom]}")


--- Clasificarea speciilor de Iris ---
Tool predictie specie Iris: setosa
Custom predictie specie Iris: setosa


In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

print("--- Validare Incrucisata ---")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
fold_idx = 1

for train_index, test_index in kf.split(X_cancer_scaled):
    # Impartim datele in Train si Test pentru fold-ul curent
    X_train, X_test = X_cancer_scaled[train_index], X_cancer_scaled[test_index]
    y_train, y_test = y_cancer[train_index], y_cancer[test_index]

    # Antrenam modelul custom
    cv_model = CustomLogisticRegression()
    cv_model.fit(X_train.tolist(), y_train.tolist(), learningRate=0.1, noEpochs=500)

    # Prevedem si calculam acuratetea
    predictions = cv_model.predict(X_test.tolist())
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    print(f"Fold {fold_idx} - Acuratete: {acc:.4f}")
    fold_idx += 1

print(f"\nAcuratetea medie (Cross-Validation): {np.mean(fold_accuracies):.4f}")

--- Validare Incrucisata ---
Fold 1 - Acuratete: 0.9035
Fold 2 - Acuratete: 0.9035
Fold 3 - Acuratete: 0.8860
Fold 4 - Acuratete: 0.9298
Fold 5 - Acuratete: 0.8230

Acuratetea medie (Cross-Validation): 0.8892


In [10]:
class BatchGDRegressionWithLoss:
    def __init__(self):
        self.intercept_ = 0.0
        self.coef_ = []

    def fit(self, x, y, learningRate=0.01, noEpochs=1000, loss='mse'):
        self.coef_ = [0.0 for _ in range(len(x[0]))]
        self.intercept_ = 0.0
        n_samples = len(x)

        for epoch in range(noEpochs):
            gradients_w = [0.0 for _ in range(len(x[0]))]
            gradient_b = 0.0

            for i in range(n_samples):
                ycomputed = self.eval(x[i])

                # Diferenta in calculul erorii/gradientului in functie de LOSS
                if loss == 'mse':
                    crtError = ycomputed - y[i]
                elif loss == 'mae':
                    crtError = 1.0 if (ycomputed - y[i]) > 0 else -1.0

                for j in range(len(x[0])):
                    gradients_w[j] += crtError * x[i][j]
                gradient_b += crtError

            for j in range(len(x[0])):
                self.coef_[j] -= learningRate * gradients_w[j] / n_samples
            self.intercept_ -= learningRate * gradient_b / n_samples

    def eval(self, xi):
        yi = self.intercept_
        for j in range(len(xi)):
            yi += self.coef_[j] * xi[j]
        return yi

    def predict(self, x):
        return [self.eval(xi) for xi in x]

print("--- Investigare Loss (MSE vs MAE) pe predictia fericirii ---")

# Antrenam cu MSE
reg_mse = BatchGDRegressionWithLoss()
reg_mse.fit(X_2_scaled, outputs_happiness, learningRate=0.1, noEpochs=1000, loss='mse')
preds_mse = reg_mse.predict(X_2_scaled)
print(f"Eroare (MSE) antrenat cu MSE Loss: {mean_squared_error(outputs_happiness, preds_mse):.4f}")

# Antrenam cu MAE
reg_mae = BatchGDRegressionWithLoss()
reg_mae.fit(X_2_scaled, outputs_happiness, learningRate=0.1, noEpochs=1000, loss='mae')
preds_mae = reg_mae.predict(X_2_scaled)
print(f"Eroare (MSE) antrenat cu MAE Loss: {mean_squared_error(outputs_happiness, preds_mae):.4f}")

--- Investigare Loss (MSE vs MAE) pe predictia fericirii ---
Eroare (MSE) antrenat cu MSE Loss: 0.3251
Eroare (MSE) antrenat cu MAE Loss: 0.3332


In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("--- Calitatea clasificatorului pentru diferite praguri de decizie ---")

X_train_c, X_test_c = X_cancer_scaled[:400], X_cancer_scaled[400:]
y_train_c, y_test_c = y_cancer[:400], y_cancer[400:]

model_thresh = CustomLogisticRegression()
model_thresh.fit(X_train_c.tolist(), y_train_c.tolist(), learningRate=0.1, noEpochs=1000)

praguri = [0.2, 0.5, 0.8]

for prag in praguri:
    preds = model_thresh.predict(X_test_c.tolist(), threshold=prag)

    acc = accuracy_score(y_test_c, preds)
    prec = precision_score(y_test_c, preds, zero_division=0)
    rec = recall_score(y_test_c, preds, zero_division=0)

    print(f"--- Prag {prag} ---")
    print(f"Acuratete: {acc:.3f} | Precizie: {prec:.3f} | Recall: {rec:.3f}")

--- Calitatea clasificatorului pentru diferite praguri de decizie ---
--- Prag 0.2 ---
Acuratete: 0.911 | Precizie: 0.960 | Recall: 0.923 | F1-Score: 0.941
--- Prag 0.5 ---
Acuratete: 0.852 | Precizie: 0.982 | Recall: 0.823 | F1-Score: 0.895
--- Prag 0.8 ---
Acuratete: 0.680 | Precizie: 1.000 | Recall: 0.585 | F1-Score: 0.738
